# 12. Prepare A2D2 for Stage 3 v5

- Google Drive의 91 GB front-center `.tar`를 **전체 압축 해제하지 않는다**.
- camera JSON timestamp를 스캔해 10 Hz frame만 선택한다.
- 선택된 PNG만 Colab local scratch로 decode/resize 후 60초 MP4를 만든다.
- 최종 MP4/NPZ만 Drive `DATASET/A2D2/processed`에 저장한다.
- A2D2 steering wheel angle은 comma2k19 steering과 scale이 다르므로 기존 shared steering regression에는 넣지 않고 `aux_metadata`에 보존한다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys

DRIVE_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
REPO = Path('/content/Blackbox-Detection')
BRANCH = 'stage3-sangchun'

if not REPO.exists():
    subprocess.run([
        'git','clone','-b',BRANCH,
        'https://github.com/sangchun1/Blackbox-Detection.git', str(REPO)
    ], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)
], check=True)
print(REPO)


## 먼저 패치 파일 추가

이 notebook과 함께 제공된 `src/blackbox_detection/stage3/a2d2.py`를 repo의 동일 경로에 추가하고 commit/push한 뒤 실행한다.


In [ ]:
A2D2_MODULE = REPO / 'src/blackbox_detection/stage3/a2d2.py'
assert A2D2_MODULE.is_file(), f'Missing: {A2D2_MODULE}'
print('module:', A2D2_MODULE)


In [ ]:
from blackbox_detection.stage3.a2d2 import (
    A2D2PrepareConfig, load_bus_json, audit_bus, prepare_session,
)

A2D2_ROOT = DRIVE_ROOT / 'DATASET/A2D2'
ARCHIVE_DIR = A2D2_ROOT / 'archives'
RAW_ROOT = A2D2_ROOT / 'raw'
PROCESSED_ROOT = A2D2_ROOT / 'processed'
SESSION_ID = '20190401_121727'

CAMERA_TAR = ARCHIVE_DIR / 'camera_lidar-20190401121727_camera_frontcenter.tar'
BUS_JSON = RAW_ROOT / 'camera_lidar' / SESSION_ID / 'bus' / '20190401121727_bus_signals.json'

assert CAMERA_TAR.is_file(), CAMERA_TAR
assert BUS_JSON.is_file(), BUS_JSON
print('camera GiB:', CAMERA_TAR.stat().st_size / 1024**3)
print('bus MiB   :', BUS_JSON.stat().st_size / 1024**2)


## 1. Bus sign audit


In [ ]:
bus_report = audit_bus(load_bus_json(BUS_JSON))
print(bus_report)
assert bus_report['accel_dvdt_corr'] > 0.80
assert bus_report['steer_yaw_corr'] > 0.50
print('BUS AUDIT: PASS')


## 2. Full prepare

첫 실행은 91 GB tar의 header/JSON scan과 약 9천 selected PNG decode가 필요하므로 시간이 걸린다. Drive에는 원본 PNG를 만들지 않는다.


In [ ]:
cfg = A2D2PrepareConfig(
    processed_root=PROCESSED_ROOT,
    width=512,
    height=384,
    target_fps=10,
    segment_frames=600,
    jpeg_quality=92,
    crf=23,
    ffmpeg_preset='veryfast',
    max_camera_skew_ms=25.0,
    overwrite=False,
    work_root=Path('/content/a2d2_work'),
)

report = prepare_session(
    camera_tar=CAMERA_TAR,
    bus_json=BUS_JSON,
    session_id=SESSION_ID,
    cfg=cfg,
)
report


## 3. Output audit


In [ ]:
import json, pandas as pd
from blackbox_detection.stage3.schema import read_frame_table

manifest = pd.read_csv(PROCESSED_ROOT / 'manifest.csv')
display(manifest)
print('segments:', len(manifest))
print('frames  :', int(manifest.num_frames.sum()))
print('hours   :', manifest.num_frames.sum() / 10 / 3600)

for row in manifest.itertuples(index=False):
    video = PROCESSED_ROOT / row.video_relpath
    meta = PROCESSED_ROOT / row.metadata_relpath
    aux = PROCESSED_ROOT / row.aux_metadata_relpath
    assert video.is_file(), video
    assert meta.is_file(), meta
    assert aux.is_file(), aux
    df = read_frame_table(meta)
    assert len(df) == row.num_frames
    assert df.frame_index_10hz.tolist() == list(range(len(df)))

print('OUTPUT AUDIT: PASS')
print((PROCESSED_ROOT / 'prepare_report.json').read_text())


### 생성 구조

```text
DATASET/A2D2/processed/
├── videos/a2d2_20190401_121727/000.mp4 ...
├── metadata/a2d2_20190401_121727/000.npz ...
├── aux_metadata/a2d2_20190401_121727/000.npz ...
├── manifest.csv
└── prepare_report.json
```

`metadata`는 기존 Stage3 frame schema와 호환된다. `valid_steer=False`인 것은 의도된 동작이다. v5에서 `aux_metadata`의 steering direction/magnitude, brake, accelerator를 별도 auxiliary head와 event-balanced sampler에 사용한다.
